# Binary Classifier — Insect vs Background

Trains a binary insect vs background classifier on manually annotated crops from `annotate.py`.

## Two backbones compared
| | EfficientNet-B2 | InsectNet backbone |
|---|---|---|
| Input | 256×256 px | 224×224 px |
| Pretrained | ImageNet (1.2M images) | 6M insect species images |
| Trainable | All layers | Final fc only (frozen backbone) |
| Advantage | Fast, lightweight | Insect-specific features |

**Domain shift note:** InsectNet was trained on clean, centred insect photos. Arctic field crops are noisy and motion-blurred. Freeze the backbone first (`MODEL = 'insectnet'`). If validation recall is low, unfreeze the last block with a small lr (see Cell 3).

## Workflow
1. Run `ls_v4.ipynb` batch → crops
2. Run `annotate.py` → `labeled/insect/` and `labeled/background/`
3. Run this notebook → `models/binary_best.pth`

**Minimum data:** ~500 insect crops, ~1000 background crops (include tiles and context crops, not only tight crops).


In [ ]:
import json
import time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    from sklearn.metrics import classification_report, confusion_matrix
    HAS_SKLEARN = True
except ImportError:
    print('Install sklearn: pip install scikit-learn')
    HAS_SKLEARN = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
print("Extracting...")
with zipfile.ZipFile('/content/drive/MyDrive/pollinator-classification/Insects_images/annotated_crops.zip', 'r') as z:
    z.extractall('/content/')
print("Done")

## Configuration
Change these paths and settings before running.

In [ ]:
# ── Data mode ──────────────────────────────────────────────────────────────
# 'ls'       → Lian's data only       (annotated_crops/labeled)
# 'mb'       → Marcus's data only     (annotated_crops_mb/labeled)
# 'combined' → both datasets merged
DATA_MODE = 'combined'

# ── Paths ──────────────────────────────────────────────────────────────────
# Colab: _BASE = Path('/content')
# Local: _BASE = Path('/Users/lianshi/Downloads/bachelor thesis/automated-ecological-image-analysis/ml-pipelines/notebooks/pollinator-classification')
_BASE = Path('/content')

# After unzipping annotated_crops.zip to /content/, paths are:
_LABELED_LS = _BASE / 'labeled_ls'
_LABELED_MB = _BASE / 'labeled_mb'


if DATA_MODE == 'ls':
    LABELED_DIR = _LABELED_LS
elif DATA_MODE == 'mb':
    LABELED_DIR = _LABELED_MB
elif DATA_MODE == 'combined':
    LABELED_DIR = None   # handled in load_and_split below
else:
    raise ValueError(f'Unknown DATA_MODE: {DATA_MODE}')

INSECTNET_WEIGHTS = Path('/content/drive/MyDrive/pollinator-classification/InsectNet/model.pth')    # only needed for insectnet


from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

MODEL_OUT_DIR = Path(f'/content/drive/MyDrive/pollinator-classification/models/models_{DATA_MODE}_{timestamp}')
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)


# ── Model choice ───────────────────────────────────────────────────────────
# 'efficientnet' → EfficientNet-B2, 256px, all layers trainable
# 'insectnet'    → InsectNet backbone (RegNet-Y-32GF), 224px, frozen backbone
# 'both'         → train both and compare
MODEL = 'efficientnet'

# ── Training settings ──────────────────────────────────────────────────────
EPOCHS    = 20
BATCH     = 32
LR        = 1e-3     # use 1e-5 if unfreezing backbone blocks
VAL_FRAC  = 0.2
TEST_FRAC = 0.1      # held-out test fraction (never seen during training)
SEED      = 42

# ── Sampling settings ────────────────────────────────────────────────────────
BG_RATIO  = 3        # background : insect ratio (combined insect count × BG_RATIO)

# ── Partial fine-tune (optional, use after initial frozen training) ─────────
# Set UNFREEZE_LAST_BLOCK = True to unfreeze the last backbone block + fc.
# Use a smaller LR (1e-5) to avoid destroying pretrained insect features.
# Only do this if frozen training recall is unsatisfactory.
UNFREEZE_LAST_BLOCK = True
UNFREEZE_LR = 1e-5

# ── Verify labeled data ────────────────────────────────────────────────────
insect_folders = ['bumblebee', 'fly', 'butterfly', 'other']
_dirs_to_check = [_LABELED_LS, _LABELED_MB] if DATA_MODE == 'combined' else [LABELED_DIR]

for _d in _dirs_to_check:
    print(f'\n{_d.name}:')
    for folder in insect_folders:
        d = _d / folder
        n = len(list(d.glob('*.jpg'))) + len(list(d.glob('*.png'))) if d.exists() else 0
        print(f'  {folder:12}: {n:>5}')
    bg_n = len(list((_d / 'background').glob('*.jpg'))) if (_d / 'background').exists() else 0
    total_insect = sum(
        len(list((_d / f).glob('*.jpg'))) + len(list((_d / f).glob('*.png')))
        for f in insect_folders if (_d / f).exists()
    )
    print(f'  insect total : {total_insect:>5}')
    print(f'  background   : {bg_n:>5}')


## Dataset and Splits

In [ ]:
CLASSES = ['background', 'insect']  # index 0=background, 1=insect

# Folders in labeled_dir that count as 'insect' (label index 1)
# 'unsure' is excluded from training
INSECT_FOLDERS = ['bumblebee', 'fly', 'butterfly', 'other']
torch.manual_seed(SEED)
np.random.seed(SEED)


class CropDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label


def parse_plot_species(path):
    """
    Extract (site, species, plot) key from crop filename.

    Handles two formats:
      hdd_1_2025_cg_Vamy_p1_101_WSCT__WSCT3529_crop_9_normal_roi.jpg
      hdd_2_2025_desert_Asa_p2_20250729_102_WSCT__WSCT1266_crop_2_normal_roi.jpg

    After the hdd_{n}_{year} prefix, fields are: site, species, plot, [date], camera...
    A date token looks like 8 digits starting with 202x (e.g. 20250729).
    Returns 'site_species_plot' key.
    """
    import re
    parts = path.stem.split('_')
    try:
        if parts[0] == 'hdd' and len(parts) > 6:
            # Skip: hdd(0), n(1), year(2)
            site    = parts[3]
            species = parts[4]
            plot    = parts[5]
            # parts[6] might be a date like 20250729 — skip it if so
            # (date tokens are 8-digit strings starting with 202)
            hdd = parts[1]  # hdd number (1, 2, ...)
            return f'hdd{hdd}_{site}_{species}_{plot}'
    except IndexError:
        pass
    return 'unknown'


def sample_background_balanced(bg_paths, n_total, seed):
    """
    Sample n_total background crops evenly across (site, species, plot) groups.
    bg_paths: list of Path objects — can span multiple directories.
    Each group contributes an equal quota; remainder is distributed round-robin.
    """
    # Group by plot key
    groups = {}
    for p in bg_paths:
        key = parse_plot_species(p)
        groups.setdefault(key, []).append(p)

    rng = np.random.default_rng(seed)
    for key in groups:
        rng.shuffle(groups[key])

    n_groups = len(groups)
    if n_groups == 0 or n_total == 0:
        return []

    quota     = n_total // n_groups
    remainder = n_total % n_groups

    print(f'Background sampling: {n_groups} groups, quota={quota}/group, remainder={remainder}')
    for key, imgs in sorted(groups.items()):
        print(f'  {key:25}: {len(imgs):>5} available')

    sampled = []
    keys = sorted(groups.keys())
    for i, key in enumerate(keys):
        imgs = groups[key]
        take = quota + (1 if i < remainder else 0)
        take = min(take, len(imgs))
        sampled.extend(imgs[:take])

    rng.shuffle(sampled)
    print(f'Sampled {len(sampled)} background crops (target {n_total})')
    return sampled


def load_and_split(labeled_dir, val_frac, seed, bg_ratio=None, test_frac=None):
    """
    bg_ratio:  background kept = insect_count × bg_ratio (default: BG_RATIO global).
    test_frac: held-out test fraction (default: TEST_FRAC global).
    Background is sampled evenly across (site, species, plot) groups.
    """
    if bg_ratio  is None: bg_ratio  = BG_RATIO
    if test_frac is None: test_frac = TEST_FRAC

    # Collect insect samples
    insect_paths = []
    for folder in INSECT_FOLDERS:
        d = labeled_dir / folder
        if not d.exists(): continue
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            insect_paths.extend(d.glob(ext))

    n_insect = len(insect_paths)
    n_bg_target = n_insect * bg_ratio

    # Collect and sample background
    bg_dir = labeled_dir / 'background'
    all_bg = []
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        all_bg.extend(bg_dir.glob(ext))
    bg_paths = sample_background_balanced(all_bg, n_bg_target, seed)

    all_samples = [(p, 0) for p in bg_paths] + [(p, 1) for p in insect_paths]

    rng = np.random.default_rng(seed)
    by_class = {0: [], 1: []}
    for i, (_, lbl) in enumerate(all_samples):
        by_class[lbl].append(i)

    train_idx, val_idx, test_idx = [], [], []
    for lbl, idxs in by_class.items():
        idxs = list(idxs)
        rng.shuffle(idxs)
        n_test = max(1, int(len(idxs) * test_frac))
        n_val  = max(1, int(len(idxs) * val_frac))
        test_idx.extend(idxs[:n_test])
        val_idx.extend(idxs[n_test:n_test + n_val])
        train_idx.extend(idxs[n_test + n_val:])

    counts = {0: len(by_class[0]), 1: len(by_class[1])}
    ratio = counts[0] / max(1, counts[1])

    # Count per-class in each split
    def split_counts(idx):
        bg = sum(1 for i in idx if all_samples[i][1] == 0)
        ins = sum(1 for i in idx if all_samples[i][1] == 1)
        return bg, ins

    tr_bg, tr_ins = split_counts(train_idx)
    va_bg, va_ins = split_counts(val_idx)
    te_bg, te_ins = split_counts(test_idx)

    print(f'\nDataset split summary:')
    print(f'  Strategy: stratified by class, test_frac={test_frac:.0%}, val_frac={val_frac:.0%}')
    print(f'  {"Split":8}  {"Background":>12}  {"Insect":>8}  {"Total":>8}')
    print(f'  {"Train":8}  {tr_bg:>12}  {tr_ins:>8}  {len(train_idx):>8}')
    print(f'  {"Val":8}  {va_bg:>12}  {va_ins:>8}  {len(val_idx):>8}')
    print(f'  {"Test":8}  {te_bg:>12}  {te_ins:>8}  {len(test_idx):>8}')
    print(f'  {"Total":8}  {counts[0]:>12}  {counts[1]:>8}  {counts[0]+counts[1]:>8}')
    print(f'  Background:insect ratio = {ratio:.1f}:1')
    if counts[1] < 150:
        print('  WARNING: fewer than 150 insect crops — annotate more before training')
    return all_samples, train_idx, val_idx, test_idx, counts


def letterbox(img, size):
    """Pad image to square with black borders then resize. Preserves aspect ratio."""
    w, h = img.size
    max_side = max(w, h)
    sq = Image.new('RGB', (max_side, max_side), (0, 0, 0))
    sq.paste(img, ((max_side - w) // 2, (max_side - h) // 2))
    return sq.resize((size, size), Image.BILINEAR)


def make_loaders(all_samples, train_idx, val_idx, test_idx, counts, img_size, batch):
    train_tf = T.Compose([
        T.Lambda(lambda img: letterbox(img, img_size)),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomRotation(30),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        # GaussianBlur simulates motion blur common in field crops
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    val_tf = T.Compose([
        T.Lambda(lambda img: letterbox(img, img_size)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    train_samples = [all_samples[i] for i in train_idx]
    val_samples   = [all_samples[i] for i in val_idx]
    test_samples  = [all_samples[i] for i in test_idx]

    train_labels = [s[1] for s in train_samples]
    cls_w = 1.0 / np.maximum(np.bincount(train_labels, minlength=2), 1)
    sample_w = [cls_w[l] for l in train_labels]
    sampler = WeightedRandomSampler(sample_w, len(sample_w))

    train_loader = DataLoader(CropDataset(train_samples, train_tf),
                              batch_size=batch, sampler=sampler, num_workers=0)
    val_loader   = DataLoader(CropDataset(val_samples, val_tf),
                              batch_size=batch, shuffle=False, num_workers=0)
    test_loader  = DataLoader(CropDataset(test_samples, val_tf),
                              batch_size=batch, shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader


if DATA_MODE == 'combined':
    # ── Pool insects from BOTH datasets, then sample background globally ──────
    # Step 1: collect all insect paths from both datasets
    insect_paths = []
    for _dir in [_LABELED_LS, _LABELED_MB]:
        for folder in INSECT_FOLDERS:
            d = _dir / folder
            if not d.exists(): continue
            for ext in ('*.jpg', '*.jpeg', '*.png'):
                insect_paths.extend(d.glob(ext))

    n_insect    = len(insect_paths)
    n_bg_target = n_insect * BG_RATIO
    print(f'Combined insects: {n_insect}  →  background target: {n_bg_target} ({BG_RATIO}:1)')

    # Step 2: collect all background paths from BOTH datasets
    all_bg = []
    for _dir in [_LABELED_LS, _LABELED_MB]:
        bg_dir = _dir / 'background'
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            all_bg.extend(bg_dir.glob(ext))

    # Step 3: sample evenly across plots from the combined pool
    bg_paths = sample_background_balanced(all_bg, n_bg_target, SEED)

    # Step 4: build all_samples and stratified split
    all_samples = [(p, 0) for p in bg_paths] + [(p, 1) for p in insect_paths]
    rng = np.random.default_rng(SEED)
    by_class = {0: [], 1: []}
    for i, (_, lbl) in enumerate(all_samples):
        by_class[lbl].append(i)

    train_idx, val_idx, test_idx = [], [], []
    for lbl, idxs in by_class.items():
        idxs = list(idxs)
        rng.shuffle(idxs)
        n_test = max(1, int(len(idxs) * TEST_FRAC))
        n_val  = max(1, int(len(idxs) * VAL_FRAC))
        test_idx.extend(idxs[:n_test])
        val_idx.extend(idxs[n_test:n_test + n_val])
        train_idx.extend(idxs[n_test + n_val:])

    counts = {0: len(bg_paths), 1: n_insect}

    def split_counts(idx):
        bg  = sum(1 for i in idx if all_samples[i][1] == 0)
        ins = sum(1 for i in idx if all_samples[i][1] == 1)
        return bg, ins

    tr_bg, tr_ins = split_counts(train_idx)
    va_bg, va_ins = split_counts(val_idx)
    te_bg, te_ins = split_counts(test_idx)
    ratio = counts[0] / max(1, counts[1])

    print(f'\nDataset split summary (combined):')
    print(f'  Strategy: stratified by class, test_frac={TEST_FRAC:.0%}, val_frac={VAL_FRAC:.0%}')
    print(f'  {"Split":8}  {"Background":>12}  {"Insect":>8}  {"Total":>8}')
    print(f'  {"Train":8}  {tr_bg:>12}  {tr_ins:>8}  {len(train_idx):>8}')
    print(f'  {"Val":8}  {va_bg:>12}  {va_ins:>8}  {len(val_idx):>8}')
    print(f'  {"Test":8}  {te_bg:>12}  {te_ins:>8}  {len(test_idx):>8}')
    print(f'  {"Total":8}  {counts[0]:>12}  {counts[1]:>8}  {counts[0]+counts[1]:>8}')
    print(f'  Background:insect ratio = {ratio:.1f}:1')
    if counts[1] < 150:
        print('  WARNING: fewer than 150 insect crops — annotate more before training')
else:
    all_samples, train_idx, val_idx, test_idx, counts = load_and_split(LABELED_DIR, VAL_FRAC, SEED)


## Model Builders

In [ ]:
def build_efficientnet():
    """EfficientNet-B2 pretrained on ImageNet. All layers trainable.
    B2 (9.1M params, 256px) gives better feature extraction than B0
    for small, blurry Arctic field crops.
    """
    print('Building EfficientNet-B2 (ImageNet, 256px, all layers trainable)')
    model = torchvision.models.efficientnet_b2(weights='IMAGENET1K_V1')
    in_features = model.classifier[-1].in_features  # 1408 for B2
    model.classifier[-1] = nn.Linear(in_features, 2)
    n_total = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'All {n_total:,} parameters are trainable ({n_train:,} trainable)')
    print(f'Strategy: full fine-tune from ImageNet weights')
    return model, 256


def build_insectnet(weights_path, unfreeze_last_block=False):
    """
    InsectNet backbone (RegNet-Y-32GF) with frozen backbone.

    Default: freeze backbone, train only final fc.
    Good for small datasets — avoids overfitting.

    Set unfreeze_last_block=True to also unfreeze the last backbone block
    for partial fine-tuning when domain shift is large (use lr=1e-5).
    """
    print(f'Building InsectNet binary classifier (RegNet-Y-32GF backbone, 224px)')
    if not weights_path.exists():
        raise FileNotFoundError(f'InsectNet weights not found: {weights_path}')
    print(f'Loading weights: {weights_path}')

    model = torchvision.models.regnet_y_32gf()
    model.fc = nn.Linear(3712, 2526)
    state = torch.load(weights_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state['model'] if 'model' in state else state, strict=True)
    print('InsectNet weights loaded ✓')

    # Replace classifier head for binary task
    model.fc = nn.Linear(3712, 2)
    nn.init.xavier_uniform_(model.fc.weight)
    nn.init.zeros_(model.fc.bias)
    nn.init.zeros_(model.fc.bias)

    # Freeze all backbone weights
    # Freeze all backbone layers, only train the final fc head
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith('fc.')
    print('Backbone: FROZEN (only final fc layer is trainable)')

    # Optionally unfreeze last backbone block for partial fine-tuning
    if unfreeze_last_block:
        print('Unfreezing last backbone block (trunk_output.block4) + fc')
        for name, p in model.named_parameters():
            if 'trunk_output.block4' in name or name.startswith('fc.'):
                p.requires_grad = True

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {n_train:,} / {n_total:,} params '
          f'({"frozen backbone" if not unfreeze_last_block else "last block + fc unfrozen"})')
    return model, 224


## Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum = correct = total = tp = fn = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        preds = out.argmax(1)
        loss_sum += loss.item() * labels.size(0)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)
        m = (labels == 1)
        tp += (preds[m] == 1).sum().item()
        fn += (preds[m] == 0).sum().item()
    return loss_sum/total, correct/total, tp/max(1, tp+fn)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum = correct = total = tp = fp = fn = 0
    all_p, all_l = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out   = model(imgs)
        preds = out.argmax(1)
        loss_sum += criterion(out, labels).item() * labels.size(0)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)
        all_p.extend(preds.cpu().tolist())
        all_l.extend(labels.cpu().tolist())
        tp += ((preds==1)&(labels==1)).sum().item()
        fp += ((preds==1)&(labels==0)).sum().item()
        fn += ((preds==0)&(labels==1)).sum().item()
    prec = tp / max(1, tp+fp)
    rec  = tp / max(1, tp+fn)
    f1   = 2*prec*rec / max(1e-8, prec+rec)
    return {'loss': loss_sum/total, 'acc': correct/total,
            'precision': prec, 'recall': rec, 'f1': f1,
            'preds': all_p, 'labels': all_l}


def run_training(model, model_name, img_size, counts):
    """Full training loop — saves best model, plots curves, prints report."""
    train_loader, val_loader, test_loader = make_loaders(
        all_samples, train_idx, val_idx, test_idx, counts, img_size, BATCH)

    model = model.to(DEVICE)

    # Weighted loss — insect class gets higher weight
    w = torch.tensor([1/max(1,counts[0]), 1/max(1,counts[1])],
                     dtype=torch.float, device=DEVICE)
    w = w / w.sum()
    criterion = nn.CrossEntropyLoss(weight=w)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_f1   = 0.0
    ckpt_path = MODEL_OUT_DIR / f'{model_name}_binary_best.pth'
    history   = {k: [] for k in ['tr_loss','val_loss','tr_rec','val_rec',
                                  'val_prec','val_f1','val_acc']}

    print(f'\n{"="*65}')
    print(f'{model_name}  |  img={img_size}px  |  epochs={EPOCHS}  |  lr={LR}')
    print(f'{"="*65}')
    print(f'{"Ep":>3}  {"TrLoss":>7}  {"VaLoss":>7}  '
          f'{"Prec":>6}  {"Recall":>7}  {"F1":>6}  {"Acc":>6}')

    t0 = time.time()
    for ep in range(1, EPOCHS+1):
        tr_loss, tr_acc, tr_rec = train_epoch(model, train_loader,
                                              optimizer, criterion, DEVICE)
        v = eval_epoch(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history['tr_loss'].append(tr_loss)
        history['val_loss'].append(v['loss'])
        history['tr_rec'].append(tr_rec)
        history['val_rec'].append(v['recall'])
        history['val_prec'].append(v['precision'])
        history['val_f1'].append(v['f1'])
        history['val_acc'].append(v['acc'])

        star = ''
        if v['f1'] > best_f1:
            best_f1 = v['f1']
            torch.save({'epoch': ep, 'model_name': model_name,
                        'img_size': img_size, 'state_dict': model.state_dict(),
                        'val_f1': v['f1'], 'val_recall': v['recall'],
                        'val_precision': v['precision']}, ckpt_path)
            star = ' *'

        print(f'{ep:>3}  {tr_loss:>7.4f}  {v["loss"]:>7.4f}  '
              f'{v["precision"]:>6.3f}  {v["recall"]:>7.3f}  '
              f'{v["f1"]:>6.3f}  {v["acc"]:>6.3f}{star}')

    print(f'\nDone in {(time.time()-t0)/60:.1f} min  |  '
          f'Best F1: {best_f1:.3f}  |  Saved: {ckpt_path}')

    # ── Curves ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{model_name} Training', fontsize=13)
    axes[0].plot(history['tr_loss'], label='train')
    axes[0].plot(history['val_loss'], label='val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(history['tr_rec'],  label='train recall')
    axes[1].plot(history['val_rec'], label='val recall')
    axes[1].plot(history['val_prec'],label='val precision')
    axes[1].set_title('Recall / Precision (insect)'); axes[1].legend()
    axes[2].plot(history['val_f1'],  label='F1')
    axes[2].plot(history['val_acc'], label='Accuracy')
    axes[2].set_title('F1 / Accuracy'); axes[2].legend()
    plt.tight_layout()
    curve_path = MODEL_OUT_DIR / f'{model_name}_curves.png'
    plt.savefig(curve_path, dpi=100); plt.close()
    print(f'Curves saved: {curve_path}')

    # ── Final report ──────────────────────────────────────────────────────────
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['state_dict'])
    final = eval_epoch(model, val_loader, criterion, DEVICE)

    # ── Evaluate on held-out test set ──────────────────────────────────────
    test_final = eval_epoch(model, test_loader, criterion, DEVICE)
    print(f'\n--- Validation results (best checkpoint epoch {ckpt["epoch"]}) ---')
    print(f'  Val   F1={final["f1"]:.3f}  Recall={final["recall"]:.3f}  Precision={final["precision"]:.3f}  Acc={final["acc"]:.3f}')
    print(f'  Test  F1={test_final["f1"]:.3f}  Recall={test_final["recall"]:.3f}  Precision={test_final["precision"]:.3f}  Acc={test_final["acc"]:.3f}')
    print('  (Test set was never seen during training or model selection)')

    if HAS_SKLEARN:
        print('\n--- Detailed classification report (test set) ---')
        print(classification_report(test_final['labels'], test_final['preds'],
                                    target_names=CLASSES, digits=3))

        cm = confusion_matrix(test_final['labels'], test_final['preds'])
        tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
        print('--- Confusion matrix (test set) ---')
        print('  A confusion matrix shows how the model classified each crop.')
        print('  Rows = true label, Columns = predicted label.\n')

        # Plot confusion matrix
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
        plt.colorbar(im, ax=ax)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(['Predicted:\nbackground', 'Predicted:\ninsect'], fontsize=11)
        ax.set_yticklabels(['True:\nbackground', 'True:\ninsect'], fontsize=11)
        for row in range(2):
            for col in range(2):
                color = 'white' if cm[row, col] > cm.max() / 2 else 'black'
                ax.text(col, row, str(cm[row, col]), ha='center', va='center',
                        fontsize=14, fontweight='bold', color=color)
        ax.set_title(f'{model_name} — Confusion Matrix (test set)', fontsize=12)
        plt.tight_layout()
        cm_path = MODEL_OUT_DIR / f'{model_name}_confusion_matrix.png'
        plt.savefig(cm_path, dpi=120); plt.close()
        print(f'  Confusion matrix saved: {cm_path}\n')

        print(f'  True Positives  (TP) = {tp:>5}  — correctly identified insect crops')
        print(f'  True Negatives  (TN) = {tn:>5}  — correctly identified background crops')
        print(f'  False Positives (FP) = {fp:>5}  — background crops wrongly called insect')
        print(f'  False Negatives (FN) = {fn:>5}  — insect crops missed (called background)')
        print(f'\n  Recall = TP/(TP+FN) = {tp}/{tp+fn} = {tp/(tp+fn):.3f}  (fraction of real insects detected)')
        print(f'  Precision = TP/(TP+FP) = {tp}/{tp+fp} = {tp/(tp+fp):.3f}  (fraction of insect predictions that are correct)')

    results = {'model': model_name, 'img_size': img_size,
               'best_epoch': ckpt['epoch'],
               'val_f1': final['f1'], 'val_recall': final['recall'],
               'val_precision': final['precision'], 'val_acc': final['acc'],
               'test_f1': test_final['f1'], 'test_recall': test_final['recall'],
               'test_precision': test_final['precision'], 'test_acc': test_final['acc']}
    (MODEL_OUT_DIR / f'{model_name}_results.json').write_text(
        json.dumps(results, indent=2))
    return results


## Run Training
Set `MODEL` in the Config cell to `'efficientnet'`, `'insectnet'`, or `'both'`.

In [ ]:
results = {}

if MODEL in ('efficientnet', 'both'):
    model_eff, img_eff = build_efficientnet()
    results['efficientnet'] = run_training(model_eff, 'efficientnet', img_eff, counts)

if MODEL in ('insectnet', 'both'):
    if not INSECTNET_WEIGHTS.exists():
        print(f'ERROR: InsectNet weights not found at {INSECTNET_WEIGHTS}')
        print('Set INSECTNET_WEIGHTS to the correct path.')
    else:
        model_ins, img_ins = build_insectnet(INSECTNET_WEIGHTS)
        results['insectnet'] = run_training(model_ins, 'insectnet', img_ins, counts)

if MODEL == 'both' and len(results) == 2:
    print('\n' + '='*65)
    print('COMPARISON SUMMARY')
    print('='*65)
    print(f'{"Model":20} {"F1":>6}  {"Recall":>7}  {"Precision":>10}  {"Acc":>6}')
    for name, r in results.items():
        print(f'{name:20} {r["val_f1"]:>6.3f}  {r["val_recall"]:>7.3f}  '
              f'{r["val_precision"]:>10.3f}  {r["val_acc"]:>6.3f}')
    print('\nPrioritise Recall — missing a real insect is worse than a false positive.')


## Test: Run Classifier on a Single Crop
Quick sanity check — load a crop and see what the classifier predicts.

In [ ]:
def load_classifier(ckpt_path):
    """Load a saved binary classifier for inference."""
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model_name = ckpt['model_name']
    img_size   = ckpt['img_size']

    if model_name == 'efficientnet':
        model = torchvision.models.efficientnet_b2(weights=None)
        in_features = model.classifier[-1].in_features  # 1408 for B2
        model.classifier[-1] = nn.Linear(in_features, 2)
    else:
        model = torchvision.models.regnet_y_32gf()
        model.fc = nn.Linear(3712, 2)

    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    print(f'Loaded {model_name} from {ckpt_path}')
    print(f'  val F1={ckpt["val_f1"]:.3f}  recall={ckpt["val_recall"]:.3f}')
    return model, img_size


def predict_crop(model, img_size, crop_path):
    """Predict insect or background for a single crop image."""
    tf = T.Compose([
        T.Lambda(lambda img: letterbox(img, img_size)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img = Image.open(crop_path).convert('RGB')
    x   = tf(img).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    bg_prob  = probs[0].item()
    ins_prob = probs[1].item()
    pred     = 'insect' if ins_prob > bg_prob else 'background'
    print(f'Prediction: {pred}  (insect={ins_prob:.2%}  background={bg_prob:.2%})')
    return pred, ins_prob


# ── Example usage ─────────────────────────────────────────────────────────────
ckpt_path = MODEL_OUT_DIR / 'efficientnet_binary_best.pth'
if ckpt_path.exists():
    clf, img_sz = load_classifier(ckpt_path)

    # Replace with any crop path from your results folder
    test_crop = Path('Insects_images/labeled/insect').glob('*.jpg')
    test_crop = next(test_crop, None)
    if test_crop:
        print(f'Testing on: {test_crop.name}')
        predict_crop(clf, img_sz, test_crop)
    else:
        print('No crops found in labeled/insect/ — run annotate.py first')
else:
    print(f'No checkpoint found at {ckpt_path} — run the training cell first')
